In [1]:
import geopandas as gpd
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np

In [24]:
#rutas (ALEJANDRO - SSD)
manzanas_gpkg_path = r"Z:\VOCES\DATA\Tratados\NAC\INT_MZN_NAC\Mzn_Tipologías_NAC_ImpEq_BufferB.gpkg"
capa = 'Mzn_Subtipologias_UeEQ_filtrado'

# Cargar el archivo de manzanas
gdf_manzanas = gpd.read_file(manzanas_gpkg_path, layer=capa)

In [25]:
cols = ['CVE_EDU_C', 'CVE_SAL_C', 'CVE_REC_C', 'CVE_SER_C']

# Conteos por categoría
gdf_manzanas['count_S'] = (gdf_manzanas[cols] == 'S').sum(axis=1)
gdf_manzanas['count_C'] = (gdf_manzanas[cols] == 'C').sum(axis=1)
gdf_manzanas['count_D'] = (gdf_manzanas[cols] == 'D').sum(axis=1)

In [26]:
gdf_manzanas[['count_S','count_C','count_D']].sum(axis=1).value_counts()

4    938284
Name: count, dtype: int64

In [27]:
gdf_manzanas['combo_signature'] = (
    'S' + gdf_manzanas['count_S'].astype(str) + 
    '_C' + gdf_manzanas['count_C'].astype(str) + 
    '_D' + gdf_manzanas['count_D'].astype(str)
)

In [28]:
conteo_combinaciones = (
    gdf_manzanas['combo_signature']
    .value_counts()
    .reset_index()
)

conteo_combinaciones.columns = ['combinacion', 'frecuencia']
conteo_combinaciones

,combinacion,frecuencia
0,S3_C0_D1,281258
1,S2_C0_D2,184502
2,S2_C1_D1,121129
3,S3_C1_D0,96072
4,S1_C1_D2,73222
5,S1_C0_D3,49978
6,S4_C0_D0,30397
7,S0_C1_D3,28409
8,S1_C2_D1,22708
9,S0_C0_D4,18708


In [29]:
tabla_revision = (
    gdf_manzanas.groupby(['CVE_EQ_C'])
      .size()
      .reset_index(name='n')
      .sort_values('CVE_EQ_C')
)
tabla_revision.loc['Total'] = tabla_revision.sum(numeric_only=True)
tabla_revision

,CVE_EQ_C,n
0,A,30397.0
1,B,109629.0
2,C,1584.0
3,D,428047.0
4,E,368627.0
Total,NaN,938284.0


In [30]:
tabla_revision = (
    gdf_manzanas.groupby(['combo_signature', 'CVE_EQ_C'])
      .size()
      .reset_index(name='n')
      .sort_values('combo_signature')
)
tabla_revision

,combo_signature,CVE_EQ_C,n
0,S0_C0_D4,E,18708
1,S0_C1_D3,E,28409
2,S0_C2_D2,E,13808
3,S0_C3_D1,D,2952
4,S0_C4_D0,C,77
5,S1_C0_D3,E,49978
6,S1_C1_D2,E,73222
7,S1_C2_D1,D,22708
8,S1_C3_D0,C,1507
9,S2_C0_D2,E,184502


In [31]:
def asignar_clave(row):
    count_S = row['count_S']
    count_C = row['count_C']
    count_D = row['count_D']
    
    # 1️⃣ Superávit absoluto
    if count_S == 4:
        return 'A'
    
    # 2️⃣ Sin déficits (solo S y C)
    if count_D == 0:
        if (count_S == 3 and count_C == 1) or (count_S == 2 and count_C == 2):
            return 'B'
        elif count_C == 4 or (count_C == 3 and count_S == 1):
            return 'C'
    
    # 3️⃣ Déficit absoluto
    if count_D == 4:
        return 'F'
    
    # 4️⃣ Cobertura parcial (exactamente un D)
    if count_D == 1:
        return 'D'
    
    # 5️⃣ Deficiencia estructural (2 o 3 D)
    if count_D >= 2:
        return 'E'
    
    return np.nan

gdf_manzanas['CVE_EQ_C_nuevo'] = gdf_manzanas.apply(asignar_clave, axis=1)

In [34]:
tabla_revision = (
    gdf_manzanas.groupby(['combo_signature','CVE_EQ_C_nuevo'])
      .size()
      .reset_index(name='n')
      .sort_values('combo_signature')
)
tabla_revision.loc['Total'] = tabla_revision.sum(numeric_only=True)
tabla_revision

,combo_signature,CVE_EQ_C_nuevo,n
0,S0_C0_D4,F,18708.0
1,S0_C1_D3,E,28409.0
2,S0_C2_D2,E,13808.0
3,S0_C3_D1,D,2952.0
4,S0_C4_D0,C,77.0
5,S1_C0_D3,E,49978.0
6,S1_C1_D2,E,73222.0
7,S1_C2_D1,D,22708.0
8,S1_C3_D0,C,1507.0
9,S2_C0_D2,E,184502.0


In [33]:
tabla_revision = (
    gdf_manzanas.groupby(['CVE_EQ_C_nuevo'])
      .size()
      .reset_index(name='n')
      .sort_values('CVE_EQ_C_nuevo')
)
tabla_revision.loc['Total'] = tabla_revision.sum(numeric_only=True)
tabla_revision

,CVE_EQ_C_nuevo,n
0,A,30397.0
1,B,109629.0
2,C,1584.0
3,D,428047.0
4,E,349919.0
5,F,18708.0
Total,NaN,938284.0
